# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [62]:
# imports
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [63]:
# constants

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'

In [64]:
# set up environment
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = MODEL_GPT
openai = OpenAI()

API key looks good so far


In [65]:
site_url = "https://ourworldindata.org/grapher/annual-number-of-deaths-by-cause?country=~WHO_AMR"

In [66]:
links = fetch_website_links(site_url)
links

['/',
 'https://www.oxfordmartin.ox.ac.uk/global-development',
 'https://www.ox.ac.uk/',
 'https://global-change-data-lab.org/',
 '/search',
 '/latest',
 '/donate',
 'https://ourworldindata.org/key-charts-understand-covid-pandemic',
 'https://ourworldindata.org/data-insights/at-the-peak-of-the-hiv-epidemic-aids-caused-more-than-half-of-all-deaths-in-some-countries',
 'https://ourworldindata.org/causes-of-death',
 'https://ourworldindata.org/data-insights/covid-19-was-the-third-largest-cause-of-death-in-2021',
 'https://ourworldindata.org/diarrheal-diseases',
 'https://ourworldindata.org/data-insights/diarrheal-diseases-are-among-the-most-common-causes-of-death-especially-in-children',
 'https://ourworldindata.org/explore-updated-data-on-health-disease-and-mortality-around-the-world',
 'https://ourworldindata.org/profile/health',
 'https://ourworldindata.org/hiv-aids',
 'https://ourworldindata.org/data-insights/homicide-rates-have-declined-dramatically-over-the-centuries',
 'https://our

In [67]:
len(links)

326

In [147]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a conscise summary of accidental death causes (not disease related or natural causes),
Do not include Terms of Service, Privacy, email links
Do not include links that are due to disease, old age.
Do not include ones that are not a specific category, a loosely defined category is fine, but not something like 'causes of death' without even categorising the cause
Don't miss any! If in any doubt AT ALL, provide in results please
Remember, if in doubt, provide in the list
It's EXTREMELY important that we don't miss any deaths that could be considered accidental from this list
before taking the final decision whether to include, ask yourself - are you sure this couldn't be considered accidental, if it could, include in the results
{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [148]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these web links have high relevance for the topic of investigating the main causes of accidental death (not disease related or natural causes), 
respond with the full https URL in JSON format.

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [149]:
get_links = get_links_user_prompt(site_url)
print(get_links,  len(get_links))


Here is the list of links on the website https://ourworldindata.org/grapher/annual-number-of-deaths-by-cause?country=~WHO_AMR -
Please decide which of these web links have high relevance for the topic of investigating the main causes of accidental death (not disease related or natural causes), 
respond with the full https URL in JSON format.

/
https://www.oxfordmartin.ox.ac.uk/global-development
https://www.ox.ac.uk/
https://global-change-data-lab.org/
/search
/latest
/donate
https://ourworldindata.org/key-charts-understand-covid-pandemic
https://ourworldindata.org/data-insights/at-the-peak-of-the-hiv-epidemic-aids-caused-more-than-half-of-all-deaths-in-some-countries
https://ourworldindata.org/causes-of-death
https://ourworldindata.org/data-insights/covid-19-was-the-third-largest-cause-of-death-in-2021
https://ourworldindata.org/diarrheal-diseases
https://ourworldindata.org/data-insights/diarrheal-diseases-are-among-the-most-common-causes-of-death-especially-in-children
https://ourw

In [150]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(len(links['links']))
    return links

In [151]:
select_relevant_links(site_url)

17


{'links': ['https://ourworldindata.org/causes-of-death',
  'https://ourworldindata.org/grapher/road-death-rate-vs-gdp-per-capita',
  'https://ourworldindata.org/grapher/road-mortality-rate-comparison',
  'https://ourworldindata.org/grapher/death-rate-road-traffic-injuries',
  'https://ourworldindata.org/grapher/death-rate-from-road-accidents',
  'https://ourworldindata.org/grapher/death-rate-from-road-accidents-ghe',
  'https://ourworldindata.org/grapher/death-rate-from-falls',
  'https://ourworldindata.org/grapher/death-rate-from-falls-gbd',
  'https://ourworldindata.org/grapher/death-rate-from-drowning',
  'https://ourworldindata.org/grapher/death-rate-from-drowning-ghe',
  'https://ourworldindata.org/grapher/death-rate-from-fires-and-burns-who',
  'https://ourworldindata.org/grapher/fire-deaths-by-age',
  'https://ourworldindata.org/grapher/deaths-from-fires-and-burns',
  'https://ourworldindata.org/grapher/death-rate-from-poisonings',
  'https://ourworldindata.org/grapher/deaths-fr

In [152]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
   # print(f"{link_system_prompt}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [153]:
select_relevant_links(site_url)

Selecting relevant links for https://ourworldindata.org/grapher/annual-number-of-deaths-by-cause?country=~WHO_AMR by calling gpt-4o-mini


Found 27 relevant links


{'links': ['https://ourworldindata.org/causes-of-death',
  'https://ourworldindata.org/grapher/road-deaths-by-type',
  'https://ourworldindata.org/grapher/death-rate-road-traffic-injuries',
  'https://ourworldindata.org/grapher/death-rate-from-road-accidents',
  'https://ourworldindata.org/grapher/death-rate-from-road-accidents-ghe',
  'https://ourworldindata.org/grapher/death-rate-from-falls',
  'https://ourworldindata.org/grapher/death-rate-from-falls-gbd',
  'https://ourworldindata.org/grapher/fire-death-rates',
  'https://ourworldindata.org/grapher/death-rate-from-fires-and-burns-ihme',
  'https://ourworldindata.org/grapher/death-rate-from-fires-and-burns-who',
  'https://ourworldindata.org/grapher/drowning-death-rates',
  'https://ourworldindata.org/grapher/death-rate-from-drowning',
  'https://ourworldindata.org/grapher/death-rate-from-drowning-ghe',
  'https://ourworldindata.org/grapher/death-rate-from-poisoning',
  'https://ourworldindata.org/grapher/death-rate-from-poisoning-g

In [154]:
def fetch_page_and_all_relevant_links(url):
 #   contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n\n## Relevant Links:\n"  #{contents}
    for link in relevant_links['links']:
      print(link)
   #   result += f"\n\n### Link: {link['type']}\n"
      result += fetch_website_contents(link)
    return result

In [155]:
print(fetch_page_and_all_relevant_links(site_url))

Selecting relevant links for https://ourworldindata.org/grapher/annual-number-of-deaths-by-cause?country=~WHO_AMR by calling gpt-4o-mini


Found 16 relevant links
https://ourworldindata.org/causes-of-death
https://ourworldindata.org/grapher/road-deaths-by-type
https://ourworldindata.org/grapher/death-rate-road-traffic-injuries
https://ourworldindata.org/grapher/death-rate-from-road-accidents
https://ourworldindata.org/grapher/deaths-from-road-injuries
https://ourworldindata.org/grapher/deaths-from-drowning
https://ourworldindata.org/grapher/death-rate-from-drowning
https://ourworldindata.org/grapher/death-rate-from-falls
https://ourworldindata.org/grapher/death-rate-from-fires-and-burns-ihme
https://ourworldindata.org/grapher/death-rate-from-poisoning
https://ourworldindata.org/grapher/death-rate-from-venomous-animal
https://ourworldindata.org/grapher/deaths-from-fires-and-burns
https://ourworldindata.org/grapher/deaths-drug-overdoses
https://ourworldindata.org/grapher/fatal-occupational-injuries-among-employees
https://ourworldindata.org/grapher/number-of-children-that-die-from-road-injuries
https://ourworldindata.org/gr

In [157]:

summary_system_prompt = """
 You are an assistant that analyzes the contents of several relevant pages from a health website
 and creates a short, consice summary about accidental deaths around the globe. being sure to take into account prevenlance, focussing more on the bigger hitters
 Respond in markdown without code blocks.
 Include details of rates, locations, simiarities across locations and the globe in general. We are looking for the output of this to provide a useful steer for future analysis
 """


In [158]:
def get_summary_user_prompt( url):
    user_prompt = f"""
Here are the contents of its landing page and other relevant pages;
use this information to build a short summary of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [160]:
get_summary_user_prompt(site_url)

Selecting relevant links for https://ourworldindata.org/grapher/annual-number-of-deaths-by-cause?country=~WHO_AMR by calling gpt-4o-mini
Found 17 relevant links
https://ourworldindata.org/causes-of-death
https://ourworldindata.org/grapher/road-death-rate-vs-gdp-per-capita
https://ourworldindata.org/grapher/death-rate-road-traffic-injuries
https://ourworldindata.org/grapher/death-rate-from-road-accidents
https://ourworldindata.org/grapher/death-rate-from-road-accidents-for-15--to-49-year-olds
https://ourworldindata.org/grapher/road-incident-deaths-by-age
https://ourworldindata.org/grapher/number-of-children-that-die-from-road-injuries
https://ourworldindata.org/grapher/death-rate-from-falls
https://ourworldindata.org/grapher/death-rate-from-drowning
https://ourworldindata.org/grapher/death-rate-from-fires-and-burns
https://ourworldindata.org/grapher/deaths-from-drowning
https://ourworldindata.org/grapher/deaths-from-falls
https://ourworldindata.org/grapher/deaths-from-fires-and-burns
ht

"\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short summary of the company in markdown without code blocks.\n\n\n## Landing Page:\n\n\n## Relevant Links:\nCauses of Death | Our World in Data\n\nOur World\nin Data\nBrowse by topic\nData\nLatest\nResources\nAbout\nSubscribe\nDonate\nCauses of Death\nBy\nSaloni Dattani\n,\nFiona Spooner\n,\nHannah Ritchie\n,\nand\nMax Roser\nCite this work\nReuse this work\nIntroduction\nKey Insights\nResearch & Writing\nCharts\nWhat are people dying from?\nThis question is essential to guide decisions in public health, and find ways to save lives.\nMany leading causes of death receive little mainstream attention. If news reports reflected what children died from, they would say that around 1,400 young children die from diarrheal diseases, 1,000 die from malaria, and 1,900 from respiratory infections –\nevery day\n.\nThis can change. Over time, death rates from these causes have declined across the

In [163]:
def create_summary(url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": summary_system_prompt},
            {"role": "user", "content": get_summary_user_prompt( url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [164]:
create_summary( site_url )

Selecting relevant links for https://ourworldindata.org/grapher/annual-number-of-deaths-by-cause?country=~WHO_AMR by calling gpt-4o-mini


Found 18 relevant links
https://ourworldindata.org/causes-of-death
https://ourworldindata.org/grapher/road-death-rate-vs-gdp-per-capita
https://ourworldindata.org/grapher/road-deaths-by-type
https://ourworldindata.org/grapher/death-rate-road-traffic-injuries
https://ourworldindata.org/grapher/death-rate-from-road-accidents
https://ourworldindata.org/grapher/deaths-from-road-injuries
https://ourworldindata.org/grapher/number-of-deaths-from-road-injuries
https://ourworldindata.org/grapher/deaths-from-drowning
https://ourworldindata.org/grapher/death-rate-from-drowning
https://ourworldindata.org/grapher/deaths-from-falls
https://ourworldindata.org/grapher/death-rate-from-falls
https://ourworldindata.org/grapher/deaths-from-fires-and-burns
https://ourworldindata.org/grapher/death-rate-from-fires-and-burns-who
https://ourworldindata.org/grapher/deaths-due-to-drug-use
https://ourworldindata.org/grapher/deaths-from-drug-use-disorders-who
https://ourworldindata.org/grapher/suicide-rate-who-mdb

# Summary of Accidental Deaths Around the Globe

**Prevalence and Major Causes**  
Accidental deaths, particularly from road traffic incidents, constitute a significant cause of mortality worldwide. Over one million people die each year from road injuries globally, making it one of the leading causes of accidental deaths. Other accidental causes, while less quantified in this summary, include injuries from falls, drowning, poisoning, and occupational hazards.

**Global Rates and Locations**  
- Road traffic deaths are notably higher in low- and middle-income countries compared to high-income countries, correlating inversely with GDP per capita.  
- Young adults (ages 15 to 49) exhibit higher rates of death from road injuries compared to other age groups, reflecting their increased mobility and exposure.  
- Child and infant accidental deaths are substantial, with many deaths related to preventable causes, including accidents from environmental and household risks.

**Similarities Across Locations**  
- Road injuries consistently rank as a top cause of accidental death across various regions, regardless of economic status, though the absolute rates differ.  
- The risk factors influencing accidental deaths show similarities worldwide, including inadequate road safety measures, lack of enforcement of traffic laws, poor healthcare access after accidents, and behavioral factors such as alcohol use.  
- Non-communicable diseases overshadow many other causes of death, yet accidental deaths remain a critical public health challenge due to their often preventable nature.

**Prevention and Trends**  
- Death rates from many accidental causes have shown changes over time, often influenced by improvements in healthcare, safety regulations, and infrastructure.  
- Enhanced understanding of accidental death causes has motivated the development of targeted prevention strategies such as seatbelt laws, speed regulations, improved vehicle safety standards, and public awareness campaigns.  
- Data limitations persist in many countries, particularly where cause-of-death registration is incomplete, obscuring the full global picture.

**Steering Future Analysis**  
- A focus on reducing road traffic deaths, especially in young adults and in low-to-middle income settings, would significantly impact global accidental death rates.  
- Investigating underlying social and infrastructural factors contributing to accident risk can help tailor effective interventions.  
- Improved data collection and standardization worldwide are essential to accurately monitor trends and evaluate prevention efforts.

In summary, accidental deaths, with road injuries as a primary contributor, are a major public health issue worldwide. Efforts to enhance road safety, improve healthcare response, and strengthen data quality are key to reducing their prevalence.

In [ ]:
# Get gpt-4o-mini to answer, with streaming
